# Study 2: learned policies and a decentralized threshold baseline

This notebook compares the local, global, and mixed reward learners with a simple rule-based policy on the same held-out environment seeds. The rule is identical and independent in each region; it uses local observations and ignores the shared GPI observation.

## 1. Set up the current policies and evaluation seeds

In [1]:
import pandas as pd

from study_2_economic_sustainability.config import EnvironmentConfig, TrainingConfig
from study_2_economic_sustainability.environment import ACTION_NAMES
from study_2_economic_sustainability.environment import EconomicSustainabilityEnvironment
from study_2_economic_sustainability.evaluation import run_episode
from study_2_economic_sustainability.train import load_q_tables

In [2]:
REWARD_MODES = ("local", "global", "mixed")
environment_config = EnvironmentConfig()
training_config = TrainingConfig()
training_seeds = training_config.training_seeds
evaluation_seeds = training_config.evaluation_seeds
action_ids = {name: index for index, name in enumerate(ACTION_NAMES)}

print(f"Training seeds: {training_seeds}")
print(f"Evaluation seeds: {len(evaluation_seeds)} ({evaluation_seeds[0]}?{evaluation_seeds[-1]})")

Training seeds: (11, 23, 37, 41, 53)
Evaluation seeds: 20 (200000?200019)


## 2. Define the decentralized baseline

The rule uses each region's development, currency, and four local pollutant bins. It selects high cleanup above the upper pollution threshold when currency guarantees affordability, low prevention at moderate pollution, and development when pollution is low and funds guarantee affordability. Otherwise it maintains without investment.

In [3]:
class DecentralizedThresholdPolicy:
    def select_action(self, observation, greedy=True):
        development_bin = observation[0]
        currency_bin = observation[1]
        local_pollution_bins = observation[2:6]
        highest_pollution_bin = max(local_pollution_bins)

        if highest_pollution_bin >= 2:
            if currency_bin >= 1:
                return action_ids["maintain_clean_high"]
            return action_ids["maintain_prevent_low"]

        if highest_pollution_bin == 1:
            return action_ids["maintain_prevent_low"]

        if development_bin < 2 and currency_bin >= 1:
            return action_ids["develop_none"]

        return action_ids["maintain_none"]

In [4]:
baseline_agents = [DecentralizedThresholdPolicy() for _ in environment_config.regions]
trained_policies = load_q_tables(REWARD_MODES, training_seeds, training_config, "../results", notebook_number=3)

print(f"Loaded {len(trained_policies)} learned policies")
print("The fixed baseline is evaluated once because reward mode does not affect its actions.")

Loaded 15 learned policies
The fixed baseline is evaluated once because reward mode does not affect its actions.


## 3. Evaluate all policies on the same held-out starts

In [5]:
def evaluate_policy_episode(agents, reward_mode, seed, policy, training_seed=None):
    environment = EconomicSustainabilityEnvironment(reward_mode, environment_config)
    transitions, terminated, truncated = run_episode(environment, agents, seed)
    region_count = len(environment_config.regions)
    row = {
        "policy": policy,
        "reward_mode": reward_mode,
        "training_seed": training_seed,
        "evaluation_seed": seed,
        "steps": environment.step_count,
        "failure": float(terminated),
        "final_mean_edi": sum(environment.development_indices) / region_count,
        "final_currency": sum(environment.currency_balances) / region_count,
        "prevention_spending": 0.0,
        "cleanup_spending": 0.0,
        "invalid_action_percent": 0.0,
        "mean_gpi": 0.0,
    }

    for pollutant in environment_config.pollutants:
        row[f"mean_{pollutant}"] = 0.0

    for transition in transitions:
        info = transition["info"]
        row["mean_gpi"] += info["global_pollution_index"]
        row["prevention_spending"] += sum(info["prevention_costs_paid"])
        row["cleanup_spending"] += sum(info["cleanup_costs_paid"])
        row["invalid_action_percent"] += 100 * sum(info["invalid_attempts"])

        for pollutant in environment_config.pollutants:
            row[f"mean_{pollutant}"] += sum(info["pollutant_loads"][pollutant]) / region_count

    row["mean_gpi"] /= environment.step_count
    row["invalid_action_percent"] /= environment.step_count * region_count

    for pollutant in environment_config.pollutants:
        row[f"mean_{pollutant}"] /= environment.step_count

    return row

In [6]:
evaluation_rows = []

for (reward_mode, training_seed), agents in trained_policies.items():
    for evaluation_seed in evaluation_seeds:
        evaluation_rows.append(evaluate_policy_episode(agents, reward_mode, evaluation_seed, reward_mode, training_seed))

print(f"Evaluated {len(evaluation_rows)} learned-policy episodes.")

Evaluated 300 learned-policy episodes.


In [7]:
for evaluation_seed in evaluation_seeds:
    evaluation_rows.append(evaluate_policy_episode(baseline_agents, "local", evaluation_seed, "decentralized_threshold"))

evaluations = pd.DataFrame(evaluation_rows)
print(f"Total episodes: {len(evaluations)}")
print(f"Shared held-out starts: {evaluations['evaluation_seed'].nunique()}")
evaluations.head()

Total episodes: 320
Shared held-out starts: 20


,policy,reward_mode,training_seed,evaluation_seed,steps,failure,final_mean_edi,final_currency,prevention_spending,cleanup_spending,invalid_action_percent,mean_gpi,mean_air,mean_water,mean_soil,mean_noise
0,local,local,11.0,200000,200,0.0,77.5,2208.375,11510.0,19800.0,2.625,7.838992,6.438070,7.973127,11.957166,2.044380
1,local,local,11.0,200001,200,0.0,77.5,2435.375,12140.0,21315.0,3.000,7.240554,6.316295,7.311853,10.552988,1.980481
2,local,local,11.0,200002,200,0.0,77.5,2521.125,12460.0,19645.0,2.500,7.500202,6.403062,7.330140,11.429097,2.028136
3,local,local,11.0,200003,200,0.0,77.5,2429.375,11960.0,21050.0,2.875,7.467814,6.401552,7.583293,11.005868,2.008158
4,local,local,11.0,200004,200,0.0,77.5,2390.750,11150.0,20560.0,2.875,7.554858,6.498530,7.514438,11.281705,2.056152


## 4. Summarize the learned policies and baseline

In [8]:
metrics = [
    "failure",
    "mean_gpi",
    "final_mean_edi",
    "final_currency",
    "prevention_spending",
    "cleanup_spending",
    "invalid_action_percent",
    "mean_air",
    "mean_water",
    "mean_soil",
    "mean_noise",
]

baseline_by_seed = evaluations[evaluations["policy"] == "decentralized_threshold"].set_index("evaluation_seed")[metrics]
learned = evaluations[evaluations["policy"] != "decentralized_threshold"]

In [9]:
learned_seed_summary = learned.groupby(["reward_mode", "training_seed"], sort=False)[metrics].mean()
learned_summary = learned_seed_summary.groupby("reward_mode", sort=False).agg(["mean", "std"])
print("Learned policies: mean and standard deviation across the five training seeds.")
display(learned_summary.round(2))

Learned policies: mean and standard deviation across the five training seeds.


failure      mean_gpi       final_mean_edi        final_currency  \
               mean  std     mean   std           mean    std           mean   
reward_mode                                                                    
local           0.0  0.0    12.09  5.99          77.79  12.01        4843.79   
global          0.0  0.0    14.80  5.08          73.15   3.38        5681.31   
mixed           0.0  0.0    16.41  2.55          81.89   7.93        5797.22   

                     prevention_spending           ... invalid_action_percent  \
                 std                mean      std  ...                   mean   
reward_mode                                        ...                          
local        1691.94            10208.85  3441.96  ...                   6.04   
global        645.33             9128.30  4596.32  ...                   0.39   
mixed        1063.45            10406.10  6042.76  ...                   0.92   

                  mean_air       mean_water       mean_soil        mean_noise  \
              std     mean   std       mean   std      mean    std       mean   
reward_mode                                                                     
local        7.12     8.58  2.08      13.71  6.96     19.02  13.45       2.22   
global       0.41     8.96  0.61      15.24  4.23     27.40  15.50       2.46   
mixed        1.00     9.70  1.30      18.09  3.42     29.32   6.44       2.55   

                   
              std  
reward_mode        
local        0.60  
global       0.08  
mixed        0.17  

[3 rows x 22 columns]

In [10]:
baseline_summary = baseline_by_seed.agg(["mean", "std"]).T
print("Decentralized baseline: mean and standard deviation across the 20 held-out starts.")
display(baseline_summary.round(2))

Decentralized baseline: mean and standard deviation across the 20 held-out starts.


,mean,std
failure,0.00,0.00
mean_gpi,23.58,0.28
final_mean_edi,67.69,1.96
final_currency,6845.96,198.36
prevention_spending,6290.00,34.18
cleanup_spending,6740.00,480.61
invalid_action_percent,0.00,0.00
mean_air,9.44,0.15
mean_water,18.74,0.27
mean_soil,57.42,0.56


## 5. Compare paired differences on the same starts

In [11]:
paired = learned.merge(baseline_by_seed.reset_index(), on="evaluation_seed", suffixes=("", "_baseline"))

for metric in metrics:
    paired[f"{metric}_difference"] = paired[metric] - paired[f"{metric}_baseline"]

difference_columns = [f"{metric}_difference" for metric in metrics]

In [12]:
paired_by_training_seed = paired.groupby(["reward_mode", "training_seed"], sort=False)[difference_columns].mean()
paired_difference_summary = paired_by_training_seed.groupby("reward_mode", sort=False).agg(["mean", "std"])
print("Differences are learned minus baseline; negative GPI or failure is favorable.")
display(paired_difference_summary.round(2))

Differences are learned minus baseline; negative GPI or failure is favorable.


failure_difference      mean_gpi_difference        \
                          mean  std                mean   std   
reward_mode                                                     
local                      0.0  0.0              -11.49  5.99   
global                     0.0  0.0               -8.78  5.08   
mixed                      0.0  0.0               -7.18  2.55   

            final_mean_edi_difference        final_currency_difference  \
                                 mean    std                      mean   
reward_mode                                                              
local                           10.10  12.01                  -2002.17   
global                           5.46   3.38                  -1164.65   
mixed                           14.20   7.93                  -1048.73   

                     prevention_spending_difference           ...  \
                 std                           mean      std  ...   
reward_mode                                                   ...   
local        1691.94                        3918.85  3441.96  ...   
global        645.33                        2838.30  4596.32  ...   
mixed        1063.45                        4116.10  6042.76  ...   

            invalid_action_percent_difference       mean_air_difference        \
                                         mean   std                mean   std   
reward_mode                                                                     
local                                    6.04  7.12               -0.86  2.08   
global                                   0.39  0.41               -0.48  0.61   
mixed                                    0.92  1.00                0.26  1.30   

            mean_water_difference       mean_soil_difference         \
                             mean   std                 mean    std   
reward_mode                                                           
local                       -5.03  6.96               -38.40  13.45   
global                      -3.50  4.23               -30.02  15.50   
mixed                       -0.65  3.42               -28.10   6.44   

            mean_noise_difference        
                             mean   std  
reward_mode                              
local                       -0.81  0.60  
global                      -0.56  0.08  
mixed                       -0.47  0.17  

[3 rows x 22 columns]

In [13]:
results_path = "../results/decentralized_baseline_comparison.csv"
evaluations.to_csv(results_path, index=False)
print(f"Saved episode-level results to {results_path}")

Saved episode-level results to ../results/decentralized_baseline_comparison.csv


## Preliminary interpretation

Across the 20 held-out starts, the baseline mean GPI was 23.58, compared with 12.09 for local, 14.80 for global, and 16.41 for mixed. All policies had zero failures. Each of the 15 learned training runs had a lower mean GPI than the baseline average.

The baseline spent 13,030 on prevention and cleanup combined. The learned-policy averages were about 22,217 (local), 18,357 (global), and 19,105 (mixed). Learned policies had higher final mean EDI, while the baseline retained more currency. Mean soil load was 57.42 for the baseline, compared with 19.02, 27.40, and 29.32 for local, global, and mixed. The observed pollution reduction therefore came with higher spending in these runs.

This comparison shows an advantage over one simple heuristic; it does not identify coordination as the cause. The policies also differ in reward signal and decision rule. Controlled ablations of shared information and reward sharing are needed to isolate coordination.